In [ ]:
# ── COMPARISON: FRED vs ECB CORRELATION MATRICES ────────────
# Run this AFTER you have both notebooks executed.
# This cell manually reconstructs the FRED correlations
# using the same countries and period for a fair comparison.

from fredapi import Fred

FRED_API_KEY = "your_key_here"
fred = Fred(api_key=FRED_API_KEY)

FRED_SERIES = {
    "Germany":    "IRLTLT01DEM156N",
    "Austria":    "IRLTLT01ATM156N",
    "Poland":     "IRLTLT01PLM156N",
    "Czech Rep.": "IRLTLT01CZM156N",
    "Hungary":    "IRLTLT01HUM156N",
    "Slovakia":   "IRLTLT01SKM156N",
    "Bulgaria":   "IRLTLT01BGM156N",
    "Croatia":    "IRLTLT01HRM156N",
    "Romania":    "IRLTLT01ROM156N",
}

# Fetch FRED data using same period as ECB
start = df_clean.index[0].strftime("%Y-%m-%d")   # match ECB start date exactly
raw_fred = {}
for country, sid in FRED_SERIES.items():
    try:
        s = fred.get_series(sid, observation_start=start)
        raw_fred[country] = s
        print(f"  ✓  {country}")
    except Exception as e:
        print(f"  ✗  {country}: {e}")

df_fred = pd.DataFrame(raw_fred).dropna()
df_fred.index = df_fred.index.to_period("M").to_timestamp()

# Align to same dates as ECB
common_dates = df_clean.index.intersection(df_fred.index)
df_ecb_aligned  = df_clean.loc[common_dates]
df_fred_aligned = df_fred.loc[common_dates]

# Reorder columns to match
cols = list(df_ecb_aligned.columns)
df_fred_aligned = df_fred_aligned[cols]

# Compute both correlation matrices on identical periods
corr_ecb  = df_ecb_aligned.diff().dropna().corr()
corr_fred = df_fred_aligned.diff().dropna().corr()
diff      = corr_ecb - corr_fred   # difference matrix

# ── PLOT: THREE HEATMAPS SIDE BY SIDE ───────────────────────
fig, axes = plt.subplots(1, 3, figsize=(22, 7))
fig.patch.set_facecolor("#F8F8F8")
fig.suptitle(
    "ECB vs FRED  |  Correlation of 10Y Bond Yield Changes  |  Same Period",
    fontsize=13, fontweight="bold", y=1.01, color="#1A1A2E"
)

mask = np.triu(np.ones_like(corr_ecb, dtype=bool), k=1)

# ECB
sns.heatmap(corr_ecb,  ax=axes[0], mask=mask, annot=True, fmt=".2f",
            cmap="RdYlGn", vmin=-1, vmax=1, linewidths=0.8,
            linecolor="white", annot_kws={"size": 8, "weight": "bold"},
            cbar_kws={"shrink": 0.75})
axes[0].set_title("ECB Data", fontsize=12, pad=10, color="#1A1A2E")
axes[0].tick_params(axis="x", rotation=40, labelsize=8)
axes[0].tick_params(axis="y", rotation=0,  labelsize=8)

# FRED
sns.heatmap(corr_fred, ax=axes[1], mask=mask, annot=True, fmt=".2f",
            cmap="RdYlGn", vmin=-1, vmax=1, linewidths=0.8,
            linecolor="white", annot_kws={"size": 8, "weight": "bold"},
            cbar_kws={"shrink": 0.75})
axes[1].set_title("FRED Data", fontsize=12, pad=10, color="#1A1A2E")
axes[1].tick_params(axis="x", rotation=40, labelsize=8)
axes[1].tick_params(axis="y", rotation=0,  labelsize=8)

# Difference (ECB minus FRED)
sns.heatmap(diff, ax=axes[2], mask=mask, annot=True, fmt=".2f",
            cmap="RdBu", vmin=-0.3, vmax=0.3, linewidths=0.8,
            linecolor="white", annot_kws={"size": 8, "weight": "bold"},
            cbar_kws={"shrink": 0.75, "label": "ECB minus FRED"})
axes[2].set_title("Difference (ECB − FRED)", fontsize=12, pad=10, color="#1A1A2E")
axes[2].tick_params(axis="x", rotation=40, labelsize=8)
axes[2].tick_params(axis="y", rotation=0,  labelsize=8)

plt.tight_layout()
plt.savefig("cee_ecb_vs_fred_comparison.png", dpi=150,
            bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()
print("✓ Saved: cee_ecb_vs_fred_comparison.png")

# ── PRINT LARGEST DIFFERENCES ───────────────────────────────
print("\nLargest differences between ECB and FRED (same pair):")
diff_flat = []
countries = list(corr_ecb.columns)
for i in range(len(countries)):
    for j in range(i+1, len(countries)):
        a, b = countries[i], countries[j]
        diff_flat.append((a, b, abs(diff.loc[a, b]), diff.loc[a, b]))
diff_flat.sort(key=lambda x: x[2], reverse=True)
for a, b, absd, d in diff_flat[:5]:
    print(f"  {a} ↔ {b:<14}  Δr = {d:+.3f}")

print("\nIf differences are < 0.05 — both sources are consistent.")
print("If differences are > 0.10 — worth investigating why.")